# 3. Rendimiento en NVIDIA Jetson Orin Nano
Si los notebooks 01 y 02 se ejecutaron en esta misma placa, dejar `TRANSFER = None`: los runs y las trazas locales son la referencia y no se necesita bundle. Si los modelos se entrenaron en otro equipo, indicar la ruta de su bundle privado.


In [ ]:
from pathlib import Path
import json
import platform
import sys
import pandas as pd
from shared.contract import ROOT, validate_splits
from shared.reporting.edge_bundle import verify, install
assert Path.cwd().resolve() == ROOT
assert sys.platform.startswith('linux')
print(platform.platform(), platform.machine(), sys.version)
TRANSFER = None  # Para un bundle externo: Path('/ruta/privada/al/bundle')
LIGHTGBM_RUN = None  # Si hay varios runs: Path('experiments/runs/lightgbm_night_v1/ID')
LSTM_RUN = None      # Si hay varios runs: Path('experiments/runs/lstm_causal_v1/ID')


## Selección y verificación
En modo directo se exige el informe emparejado del notebook 02. Si existe más de un run de un enfoque, fijar su ruta explícitamente arriba. En modo bundle se verifican e instalan archivos ignorados sin sobrescribir ninguno distinto.


In [ ]:
if TRANSFER is None:
    contract = validate_splits()
    quality = json.loads((ROOT / 'results' / 'metrics' / 'paired_test.json').read_text(encoding='utf-8'))
    assert quality['paired_test_rows'] == 31680
    assert all(quality[name]['data_manifest_id'] == contract['manifest_id'] for name in ('lightgbm', 'lstm'))
    def select_run(kind, selected):
        if selected is not None:
            return (ROOT / selected).resolve()
        choices = sorted((ROOT / 'experiments' / 'runs' / kind).glob('*/manifest.json'))
        assert len(choices) == 1, f'{kind}: elegir una ruta explícita entre {len(choices)} runs'
        return choices[0].parent
    lightgbm_run = select_run('lightgbm_night_v1', LIGHTGBM_RUN)
    lstm_run = select_run('lstm_causal_v1', LSTM_RUN)
else:
    plan = verify(TRANSFER)
    install(TRANSFER)
    assert validate_splits()['manifest_id'] == plan['data_manifest_id']
    runs = json.loads((TRANSFER / 'bundle_manifest.json').read_text(encoding='utf-8'))['runs']
    lightgbm_run, lstm_run = ROOT / runs['lightgbm'], ROOT / runs['lstm']
print(lightgbm_run, lstm_run, sep='\n')


## Paridad de inferencia antes del cronómetro
Se cargan primero las trazas de referencia en memoria; luego se vuelve a inferir y se comprueba que las predicciones no cambian. En modo directo, las referencias proceden del notebook 01 y del informe validado por el notebook 02.


In [ ]:
from approaches.lightgbm.night_context import export_test as export_lightgbm
from approaches.lstm.adapter import export_test as export_lstm
reference_root = ROOT / ('results/traces' if TRANSFER is None else 'experiments/runs/edge_reference')
expected = {name: pd.read_parquet(reference_root / f'{name}_test.parquet') for name in ('lightgbm_night', 'lstm')}
assert len(expected['lightgbm_night']) == len(expected['lstm']) == 31680
assert set(expected['lightgbm_night'].model_version) == {f'lightgbm-night-context-v1-{lightgbm_run.name}'}
assert set(expected['lstm'].model_version) == {f'lstm-causal-v1-{lstm_run.name}'}
if TRANSFER is None:
    assert quality['lightgbm']['model_version'] == expected['lightgbm_night'].model_version.iloc[0]
    assert quality['lstm']['model_version'] == expected['lstm'].model_version.iloc[0]
for name, produced in [('lightgbm_night', export_lightgbm(lightgbm_run)[0]), ('lstm', export_lstm(lstm_run)[0])]:
    actual = pd.read_parquet(produced)
    columns = ['timestamp', 'ground_truth_type', 'binary_prediction', 'stage_2_prediction', 'final_prediction']
    assert actual[columns].equals(expected[name][columns]), f'{name}: la inferencia cambió'
    print(name, len(actual), 'objetivos idénticos')


## Medición
Ambos benchmarks registran plataforma, versiones, calentamiento, repeticiones, unidad y alcance. LightGBM incluye extracción causal y reglas; LSTM usa ventanas ya preparadas. Mostrar ambos alcances y no interpretar sus latencias como equivalentes.


In [ ]:
from approaches.lightgbm.night_benchmark import run_benchmark as benchmark_lightgbm
from approaches.lstm.benchmark import run_benchmark as benchmark_lstm
lightgbm_report = benchmark_lightgbm(lightgbm_run, rows=2000, repeats=5)
lstm_report = benchmark_lstm(lstm_run, rows=2000, repeats=5)
for path in (lightgbm_report, lstm_report):
    report = json.loads(path.read_text(encoding='utf-8'))
    print(path, report['prediction_unit'], report['timing_scope'], report['median_latency_ms'], report['p95_latency_ms'])


Los informes quedan en `results/reports/`, ignorados por Git. Registrar además JetPack, modo de potencia, frecuencia y temperatura durante cada medición si se usan para interpretar diferencias; no aparecen todavía en el esquema común.
